In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

In [ ]:
!pip install mrmr_selection causal-learn torch networkx pandas numpy scikit-learn

In [ ]:
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from sklearn.preprocessing import StandardScaler
from sklearn.impute import SimpleImputer
from mrmr import mrmr_classif
from causallearn.search.ConstraintBased.PC import pc
from causallearn.utils.GraphUtils import GraphUtils
import networkx as nx

In [ ]:
#uploading datasets
from pathlib import Path
import pandas as pd

datasets = {
    'sixca': '/kaggle/input/datasets/golam2002/sixca-co',
    'tcga':  '/kaggle/input/datasets/golam2002/tcga-luad'
}

data = {}

for ds_name, ds_path in datasets.items():
    data[ds_name] = {}
    for file_path in Path(ds_path).rglob('*'):
        if file_path.is_file():
            try:
                data[ds_name][file_path.name] = pd.read_csv(file_path, sep="\t", index_col=0)
            except Exception as e:
                print(f"Skipped {file_path.name}: {e}")



In [ ]:
# Access example:
star_counts = data['tcga']['TCGA-LUAD.star_counts.tsv']
star_counts.sample(20)

In [ ]:
#dimesions
print(star_counts.shape)

In [ ]:
import pandas as pd
import numpy as np

def clean_ensembl_ids(gene_list):
    """Strips version tags from Ensembl IDs (e.g., ENSG00000141510.16 -> ENSG00000141510)."""
    cleaned = []
    for g in gene_list:
        g_str = str(g).strip().split('.')[0].upper()
        cleaned.append(g_str)
    return cleaned

import pandas as pd
import numpy as np

def prepare_combined_dataset(data_dict, sixca_file, tcga_file, annot_file='Human.GRCh38.p13.annot.tsv'):
    print("==================================================")
    print(" MERGING SIXCA AND TCGA DATASETS")
    print("==================================================")
    
    # 1. Load DataFrames
    df_sixca = data_dict['sixca'][sixca_file].copy()
    df_tcga = data_dict['tcga'][tcga_file].copy()
    
    # 2. Transpose matrices (Samples as rows, Genes as columns)
    X_sixca = df_sixca.T if 'Target' not in df_sixca.columns else df_sixca.drop(columns=['Target'])
    X_tcga = df_tcga.T if 'Target' not in df_tcga.columns else df_tcga.drop(columns=['Target'])

    # 3. Clean TCGA Gene IDs (ENSG00000000003.15 -> ENSG00000000003)
    X_tcga.columns = [str(c).split('.')[0].strip() for c in X_tcga.columns]
    
    # 4. Map SIXCA NCBI GeneIDs to Ensembl IDs
    annot_df = data_dict['sixca'][annot_file].copy()
    
    # Ensure GeneID is accessible as a column
    if 'GeneID' not in annot_df.columns:
        annot_df = annot_df.reset_index()
        
    # Drop rows without an Ensembl mapping
    annot_clean = annot_df.dropna(subset=['EnsemblGeneID']).astype(str)
    
    # Build GeneID -> EnsemblID dictionary
    gene_id_to_ensg = dict(
        zip(
            annot_clean['GeneID'].str.strip(), 
            annot_clean['EnsemblGeneID'].str.split('.').str[0].str.strip()
        )
    )
    
    # Map SIXCA column headers
    X_sixca.columns = [gene_id_to_ensg.get(str(col).strip(), str(col).strip()) for col in X_sixca.columns]
    
    # 5. Clean, deduplicate, and convert to numeric
    X_sixca = X_sixca.apply(pd.to_numeric, errors='coerce').fillna(0)
    X_tcga = X_tcga.apply(pd.to_numeric, errors='coerce').fillna(0)
    
    X_sixca = X_sixca.loc[:, ~X_sixca.columns.duplicated()]
    X_tcga = X_tcga.loc[:, ~X_tcga.columns.duplicated()]

    # 6. Intersect Common Ensembl Genes
    common_genes = sorted(list(set(X_sixca.columns).intersection(set(X_tcga.columns))))
    common_genes = [g for g in common_genes if g.startswith('ENSG')]
    
    print(f"-> Successfully mapped and matched {len(common_genes)} common Ensembl genes.")
    
    if len(common_genes) == 0:
        raise ValueError("No common genes found after mapping.")

    # Subset shared genes
    X_sixca_sub = X_sixca[common_genes]
    X_tcga_sub = X_tcga[common_genes]

    # 7. Target Labels (TCGA: 1=Tumor, 0=Normal | SIXCA: 1=Tumor)
    sample_ids_tcga = X_tcga_sub.index.astype(str)
    y_tcga = sample_ids_tcga.str.extract(r'TCGA-\w+-\w+-(0[1-9]|1[0-9])')[0].apply(
        lambda code: 1 if str(code).startswith('0') else 0
    )
    y_tcga.index = X_tcga_sub.index
    y_sixca = pd.Series([1] * len(X_sixca_sub), index=X_sixca_sub.index)

    # 8. Concatenate Datasets
    X_combined = pd.concat([X_sixca_sub, X_tcga_sub], axis=0)
    y_combined = pd.concat([y_sixca, y_tcga], axis=0)
    y_combined.name = 'Target'

    print(f"-> Merged Dataset Ready: {X_combined.shape[0]} total samples across SIXCA & TCGA.")
    return X_combined, y_combined

In [ ]:
# ==========================================
# DEFINE ALL PIPELINE FUNCTIONS (Run this first)
# ==========================================
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from sklearn.preprocessing import StandardScaler
from sklearn.impute import SimpleImputer
from mrmr import mrmr_classif
from causallearn.search.ConstraintBased.PC import pc

# Step 2: Preprocessing
def preprocess_data(X):
    """Handles missing values and normalizes features."""
    imputer = SimpleImputer(strategy='mean')
    X_imputed = pd.DataFrame(imputer.fit_transform(X), columns=X.columns, index=X.index)
    
    scaler = StandardScaler()
    X_scaled = pd.DataFrame(scaler.fit_transform(X_imputed), columns=X.columns, index=X.index)
    return X_scaled

# Step 3: mRMR Feature Selection
def mrmr_feature_selection(X, y, k=50):
    """Selects top k features using mRMR algorithm."""
    print(f"  -> Running mRMR on {X.shape[1]} genes to select top {k} candidate features...")
    selected_features = mrmr_classif(X=X, y=y, K=k)
    return X[selected_features]

# Step 4: Causal Skeleton & Superset MB
def find_superset_mb_pc(X_subset, y):
    """Finds causal skeleton using PC algorithm and extracts superset MB."""
    print("  -> Running PC algorithm to find Causal Skeleton...")
    data = X_subset.copy()
    data['Target'] = y.values
    data_np = data.to_numpy()
    
    # Run PC Algorithm
    cg = pc(data_np, alpha=0.05, indep_test='fisherz', verbose=False)
    adj_matrix = cg.G.graph
    
    target_idx = data.columns.get_loc('Target')
    num_nodes = adj_matrix.shape[0]
    
    # Extract 1-hop and 2-hop neighbors
    one_hop = set()
    for i in range(num_nodes):
        if adj_matrix[i, target_idx] != 0 or adj_matrix[target_idx, i] != 0:
            one_hop.add(i)
            
    superset_mb_indices = set(one_hop)
    for node in one_hop:
        for j in range(num_nodes):
            if adj_matrix[j, node] != 0 or adj_matrix[node, j] != 0:
                superset_mb_indices.add(j)
                
    if target_idx in superset_mb_indices:
        superset_mb_indices.remove(target_idx)
        
    superset_features = [data.columns[i] for i in superset_mb_indices]
    print(f"  -> Discovered Superset MB size: {len(superset_features)} features.")
    return data[superset_features]

# ==========================================
# Fixed Step 5 & 6: DAG-GNN & Biomarker Identification
# ==========================================
class SimpleDAGGNN(nn.Module):
    def __init__(self, num_nodes, hidden_dim=64):
        super(SimpleDAGGNN, self).__init__()
        self.num_nodes = num_nodes
        
        # Continuous Adjacency Matrix A (num_nodes x num_nodes)
        self.A = nn.Parameter(torch.zeros(num_nodes, num_nodes)) 
        
        # Encoder: Maps (N, num_nodes) -> (N, hidden_dim) -> (N, num_nodes)
        self.encoder = nn.Sequential(
            nn.Linear(num_nodes, hidden_dim),
            nn.ReLU(),
            nn.Linear(hidden_dim, num_nodes)
        )
        
        # Decoder: Maps (N, num_nodes) -> (N, hidden_dim) -> (N, num_nodes)
        self.decoder = nn.Sequential(
            nn.Linear(num_nodes, hidden_dim),
            nn.ReLU(),
            nn.Linear(hidden_dim, num_nodes)
        )
        
    def forward(self, x):
        I = torch.eye(self.num_nodes, device=x.device)
        M = I - self.A.t()  # Shape: (num_nodes, num_nodes)
        
        # 1. Graph Filter Transformation: (N, num_nodes) @ (num_nodes, num_nodes) -> (N, num_nodes)
        x_graph = torch.matmul(x, M)
        
        # 2. Non-linear Encoding
        z = self.encoder(x_graph)
        
        # 3. Decoding / Reconstruction
        x_hat = self.decoder(z) 
        return x_hat, self.A


def learn_mb_and_identify_biomarkers(X_superset, y, epochs=200):
    print("  -> Running DAG-GNN structure learning...")
    data = X_superset.copy()
    if 'Target' not in data.columns:
        data['Target'] = y.values
    
    tensor_data = torch.tensor(data.values, dtype=torch.float32)
    num_nodes = data.shape[1]
    
    model = SimpleDAGGNN(num_nodes=num_nodes)
    optimizer = torch.optim.Adam(model.parameters(), lr=0.01)
    mse_loss = nn.MSELoss()
    
    for epoch in range(epochs):
        optimizer.zero_grad()
        x_hat, A = model(tensor_data)
        # Reduced L1 penalty weight from 0.01 to 0.001
        loss = mse_loss(x_hat, tensor_data) + 0.0001 * torch.sum(torch.abs(A))
        loss.backward()
        optimizer.step()
        
    final_adj = model.A.detach().numpy()
    
   # Slightly lower edge threshold snippet in learn_mb_and_identify_biomarkers:
    final_adj[np.abs(final_adj) < 0.001] = 0
    
    target_idx = data.columns.get_loc('Target')
    biomarkers = set()
    
    for i in range(len(final_adj)):
        if (final_adj[i, target_idx] != 0 or final_adj[target_idx, i] != 0) and i != target_idx:
            biomarkers.add(data.columns[i])
            
    biomarker_list = list(biomarkers)
    print(f"  -> Discovered {len(biomarker_list)} Biomarkers.")
    return biomarker_list

In [ ]:
# Inspect gene identifier formats
df_sixca = data['sixca']['GSE68086_norm_counts_FPKM_GRCh38.p13_NCBI.tsv']
df_tcga = data['tcga']['TCGA-LUAD.star_counts.tsv']

print("SIXCA Gene ID samples (Index/Columns):")
print(df_sixca.index[:5] if 'Target' not in df_sixca.columns else df_sixca.columns[:5])

print("\nTCGA Gene ID samples (Index/Columns):")
print(df_tcga.index[:5] if 'Target' not in df_tcga.columns else df_tcga.columns[:5])

In [ ]:
# ==========================================
# DIAGNOSTIC CELL: Inspect Identifier Formats
# ==========================================
annot_df = data['sixca']['Human.GRCh38.p13.annot.tsv']
df_sixca = data['sixca']['GSE68086_norm_counts_FPKM_GRCh38.p13_NCBI.tsv']
df_tcga = data['tcga']['TCGA-LUAD.star_counts.tsv']

print("--- 1. ANNOTATION FILE COLUMNS ---")
print(annot_df.columns.tolist())

print("\n--- 2. ANNOTATION SAMPLE (First 2 rows) ---")
display(annot_df.head(2))

print("\n--- 3. SIXCA GENE HEADERS (First 5) ---")
sixca_genes = df_sixca.index[:5] if 'Target' not in df_sixca.columns else df_sixca.columns[:5]
print(list(sixca_genes))

print("\n--- 4. TCGA GENE HEADERS (First 5) ---")
tcga_genes = df_tcga.index[:5] if 'Target' not in df_tcga.columns else df_tcga.columns[:5]
print(list(tcga_genes))

In [ ]:
sixca_file = 'GSE68086_norm_counts_FPKM_GRCh38.p13_NCBI.tsv'
tcga_file = 'TCGA-LUAD.star_counts.tsv'

# 1. Merge SIXCA and TCGA Data
X_comb, y_comb = prepare_combined_dataset(data, sixca_file, tcga_file)

# 2. Preprocessing
X_preprocessed = preprocess_data(X_comb)

# 3. mRMR Feature Selection
X_reduced = mrmr_feature_selection(X_preprocessed, y_comb, k=50)

# 4. Causal Skeleton Discovery
X_superset = find_superset_mb_pc(X_reduced, y_comb)

# 5 & 6. DAG-GNN Structure Learning & Biomarker Extraction
final_biomarkers = learn_mb_and_identify_biomarkers(X_superset, y_comb)

print("\nFinal Discovered Biomarkers across SIXCA & TCGA:")
print(final_biomarkers)

In [ ]:
# Map Ensembl ID to HGNC Gene Symbol
annot_df = data['sixca']['Human.GRCh38.p13.annot.tsv'].copy()

# Build Ensembl -> Symbol dictionary
ens_to_symbol = dict(
    zip(
        annot_df['EnsemblGeneID'].astype(str).str.split('.').str[0].str.strip(),
        annot_df['Symbol'].astype(str)
    )
)

# Translate final biomarkers
gene_symbols = [ens_to_symbol.get(gene, gene) for gene in final_biomarkers]

print("Discovered Biomarker Gene Symbol(s):")
print(gene_symbols)

In [ ]:
#Model accuracy
import numpy as np
from sklearn.model_selection import StratifiedKFold, cross_val_score
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score

# 1. Subset combined dataset using the 31 discovered biomarkers
X_subset = X_comb[final_biomarkers]
y_subset = y_comb

# 2. Compute Acc1: Stratified 5-Fold Cross-Validation on Combined Data
rf = RandomForestClassifier(n_estimators=100, random_state=42)
acc1_scores = cross_val_score(rf, X_subset, y_subset, cv=5, scoring='accuracy')
acc1_mean = np.mean(acc1_scores)

# 3. Compute Acc2: Train on TCGA samples, Test on SIXCA samples (Domain Transfer)
tcga_mask = X_subset.index.astype(str).str.contains('TCGA')
sixca_mask = ~tcga_mask

X_train, y_train = X_subset[tcga_mask], y_subset[tcga_mask]
X_test, y_test = X_subset[sixca_mask], y_subset[sixca_mask]

rf.fit(X_train, y_train)
y_pred = rf.predict(X_test)
acc2_score = accuracy_score(y_test, y_pred)

print("==================================================")
print(" PIPELINE BENCHMARK EVALUATION RESULTS")
print("==================================================")
print(f"• Discovered Biomarkers Count : {len(final_biomarkers)}")
print(f"• Acc1 (5-Fold Cross-Validation) : {acc1_mean:.6f}")
print(f"• Acc2 (TCGA -> SIXCA Transfer)  : {acc2_score:.6f}")

In [ ]:
#Validation and Sensitivity
from sklearn.metrics import classification_report, confusion_matrix

print("Confusion Matrix (TCGA -> SIXCA Transfer):")
print(confusion_matrix(y_test, y_pred))

print("\nDetailed Classification Report:")
print(classification_report(y_test, y_pred, zero_division=0))

In [ ]:
#Step 1: Stratified Holdout Evaluation & ROC-AUC Curve
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    classification_report, confusion_matrix, roc_auc_score, 
    roc_curve, accuracy_score
)

# 1. Subset combined dataset to the 31 discovered biomarkers
X_sub = X_comb[final_biomarkers]
y_sub = y_comb

# 2. Stratified 70/30 Train/Test Split
X_train, X_test, y_train, y_test = train_test_split(
    X_sub, y_sub, test_size=0.3, stratify=y_sub, random_state=42
)

# 3. Fit Random Forest Classifier
clf = RandomForestClassifier(n_estimators=100, random_state=42)
clf.fit(X_train, y_train)

# 4. Predict probabilities and classes
y_pred = clf.predict(X_test)
y_proba = clf.predict_proba(X_test)[:, 1]

# 5. Compute Metrics
tn, fp, fn, tp = confusion_matrix(y_test, y_pred).ravel()
sensitivity = tp / (tp + fn)
specificity = tn / (tn + fp)
auc = roc_auc_score(y_test, y_proba)
acc = accuracy_score(y_test, y_pred)

print("==================================================")
print(" 70/30 STRATIFIED HOLDOUT EVALUATION RESULTS")
print("==================================================")
print(f"• Accuracy    : {acc:.4f}")
print(f"• Sensitivity : {sensitivity:.4f}")
print(f"• Specificity : {specificity:.4f}")
print(f"• ROC-AUC     : {auc:.4f}")

# 6. Plot ROC Curve
fpr, tpr, _ = roc_curve(y_test, y_proba)
plt.figure(figsize=(6, 5))
plt.plot(fpr, tpr, color='#1f77b4', lw=2, label=f'31-Biomarker Model (AUC = {auc:.4f})')
plt.plot([0, 1], [0, 1], color='gray', lw=1.5, linestyle='--')
plt.xlabel('False Positive Rate (1 - Specificity)')
plt.ylabel('True Positive Rate (Sensitivity)')
plt.title('ROC Curve - Stratified Combined Holdout Validation')
plt.legend(loc="lower right")
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

In [ ]:
#Step 2: 2D PCA Clustering & Visualization
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.decomposition import PCA

# 1. Assign granular class labels across datasets
sample_labels = []
tcga_mask = X_comb.index.astype(str).str.contains('TCGA')

for idx, is_tcga in zip(X_comb.index, tcga_mask):
    if is_tcga:
        target = y_comb.loc[idx]
        label = 'TCGA Tumor' if target == 1 else 'TCGA Normal'
    else:
        label = 'SIXCA Tumor'
    sample_labels.append(label)

# 2. Compute 2D PCA on the 31 Biomarkers
pca = PCA(n_components=2)
X_pca = pca.fit_transform(X_comb[final_biomarkers])

df_pca = pd.DataFrame(X_pca, columns=['PC1', 'PC2'])
df_pca['Group'] = sample_labels

# 3. Plot PCA Scatter Plot
plt.figure(figsize=(8, 6))
palette = {
    'TCGA Tumor': '#d62728',   # Red
    'TCGA Normal': '#1f77b4',  # Blue
    'SIXCA Tumor': '#2ca02c'   # Green
}

sns.scatterplot(
    data=df_pca, x='PC1', y='PC2', hue='Group', 
    style='Group', palette=palette, alpha=0.75, s=70
)

var_pc1 = pca.explained_variance_ratio_[0] * 100
var_pc2 = pca.explained_variance_ratio_[1] * 100

plt.title(f'PCA Projection using 31 Discovered Biomarkers\n(PC1: {var_pc1:.1f}%, PC2: {var_pc2:.1f}%)', fontsize=12)
plt.xlabel(f'Principal Component 1 ({var_pc1:.1f}%)')
plt.ylabel(f'Principal Component 2 ({var_pc2:.1f}%)')
plt.grid(True, alpha=0.3)
plt.legend(bbox_to_anchor=(1.02, 1), loc='upper left')
plt.tight_layout()
plt.show()

In [ ]:
#Expression Heatmap & Hierarchical Clustering
import seaborn as sns
import matplotlib.pyplot as plt
import pandas as pd

# 1. Z-score normalize expression across the 31 biomarkers
X_heat = X_comb[final_biomarkers].copy()
X_heat_z = (X_heat - X_heat.mean(axis=0)) / X_heat.std(axis=0)

# 2. Map sample indices to cohort labels
tcga_mask = X_comb.index.astype(str).str.contains('TCGA')
sample_labels = [
    'TCGA Tumor' if is_tcga and y_comb.loc[idx] == 1
    else ('TCGA Normal' if is_tcga else 'SIXCA Tumor')
    for idx, is_tcga in zip(X_comb.index, tcga_mask)
]

# 3. Assign row annotation colors
color_map = {'TCGA Tumor': '#d62728', 'TCGA Normal': '#1f77b4', 'SIXCA Tumor': '#2ca02c'}
row_colors = pd.Series(sample_labels, index=X_heat.index).map(color_map)

# Use Gene Symbols for column labels if available
gene_labels = [
    annot_lookup.get(g, {}).get('Symbol', g) if 'annot_lookup' in locals() else g 
    for g in final_biomarkers
]
X_heat_z.columns = gene_labels

# 4. Generate Hierarchical Clustermap
g = sns.clustermap(
    X_heat_z,
    row_colors=row_colors,
    cmap='vlag',
    figsize=(12, 9),
    yticklabels=False,
    xticklabels=True,
    cbar_kws={'label': 'Expression Z-Score'}
)

g.ax_heatmap.set_xticklabels(g.ax_heatmap.get_xticklabels(), rotation=90, fontsize=9)
g.fig.suptitle("Hierarchical Clustering of 31 Discovered Biomarkers Across Cohorts", y=1.02, fontsize=13)
plt.show()

In [ ]:
#Pathway Enrichment Analysis (KEGG & GO)
!pip install gseapy mygene -q

import gseapy as gp
import mygene
import pandas as pd

# 1. Map Ensembl IDs to official HGNC Gene Symbols using MyGeneInfo
mg = mygene.MyGeneInfo()
gene_info = mg.querymany(
    final_biomarkers, 
    scopes='ensembl.gene', 
    fields='symbol', 
    species='human', 
    as_dataframe=False
)

# 2. Extract valid mapped symbols
mapped_symbols = list(set([
    entry['symbol'] for entry in gene_info if 'symbol' in entry
]))

print(f"Successfully mapped {len(mapped_symbols)} / {len(final_biomarkers)} biomarkers to HGNC Symbols.")
print("Mapped Gene Symbols:", mapped_symbols)

# 3. Run Enrichr analysis on HGNC Symbols
enr = gp.enrichr(
    gene_list=mapped_symbols,
    gene_sets=['KEGG_2021_Human', 'GO_Biological_Process_2023'],
    organism='human',
    outdir=None
)

# 4. Display and export enriched pathways
df_enrich = enr.results.sort_values('Adjusted P-value')
print("\nTop 10 Enriched Pathways / Biological Processes:")
display(df_enrich[['Gene_set', 'Term', 'Adjusted P-value', 'Overlap', 'Genes']].head(10))

df_enrich.to_csv('pathway_enrichment_results.csv', index=False)
print("\nEnrichment results saved to 'pathway_enrichment_results.csv'.")

In [ ]:
#Ablation Benchmark Study (Causal vs. Non-Causal Baseline)
import numpy as np
import pandas as pd
from sklearn.model_selection import cross_val_score
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LassoCV
from sklearn.metrics import accuracy_score

def evaluate_feature_set(feature_cols, method_name):
    X_sub = X_comb[feature_cols]
    y_sub = y_comb
    
    # Acc1: Stratified 5-Fold Cross Validation
    rf = RandomForestClassifier(n_estimators=100, random_state=42)
    acc1 = np.mean(cross_val_score(rf, X_sub, y_sub, cv=5, scoring='accuracy'))
    
    # Acc2: Domain Transfer (Train on TCGA, Test on SIXCA)
    tcga_mask = X_sub.index.astype(str).str.contains('TCGA')
    X_tr, y_tr = X_sub[tcga_mask], y_sub[tcga_mask]
    X_te, y_te = X_sub[~tcga_mask], y_sub[~tcga_mask]
    
    rf.fit(X_tr, y_tr)
    acc2 = accuracy_score(y_te, rf.predict(X_te))
    
    return {
        'Feature Selection Method': method_name,
        '# Genes': len(feature_cols),
        'Acc1 (Cross-Val)': round(acc1, 6),
        'Acc2 (Domain Transfer)': round(acc2, 6)
    }

print("Running Ablation Study Comparisons...")

# 1. Baseline A: Top 31 Highest Variance Genes
top_var_genes = X_comb.var(axis=0).nlargest(31).index.tolist()

# 2. Baseline B: Top 31 Lasso Selected Genes (Trained on TCGA)
tcga_mask = X_comb.index.astype(str).str.contains('TCGA')
lasso = LassoCV(cv=3, random_state=42, max_iter=2000).fit(X_comb.loc[tcga_mask], y_comb[tcga_mask])
lasso_coefs = pd.Series(np.abs(lasso.coef_), index=X_comb.columns)
lasso_genes = lasso_coefs.nlargest(31).index.tolist()

# 3. Proposed: Causal Markov Blanket (mRMR + PC + DAG-GNN)
causal_genes = final_biomarkers

# Execute evaluation suite
ablation_results = [
    evaluate_feature_set(top_var_genes, 'Top 31 Variance Genes'),
    evaluate_feature_set(lasso_genes, 'Lasso (L1 Regularization)'),
    evaluate_feature_set(causal_genes, 'Ours (mRMR + PC + DAG-GNN)')
]

df_ablation = pd.DataFrame(ablation_results)

print("\n==================================================")
print(" ABLATION EXPERIMENT BENCHMARK SUMMARY")
print("==================================================")
display(df_ablation)

df_ablation.to_csv('ablation_study_results.csv', index=False)